# NBA Comp Viz — Colab harvest runner
Runs the full harvest chain (build → segment → matchups → align → join → credit)
on a Colab GPU (~4-6× faster than local MPS). The repo is self-contained: code,
model weights, PBP data and the games registry all clone from GitHub.

**Runtime → Change runtime type → GPU (T4)** before running.

Resume-safe: `data/harvest/status.json` is kept on Drive, so a killed session
picks up where it left off — just Run All again.

In [ ]:
#@title 1 · Clone repo + install deps + PREFLIGHT
!git clone --depth 1 https://github.com/lmcnulty7/NBA-Comp-Viz-V3.git repo 2>/dev/null || (cd repo && git pull)
%cd repo
!pip -q install ultralytics easyocr yt-dlp 2>&1 | tail -1
import torch, os
print("device:", "cuda" if torch.cuda.is_available() else "CPU ONLY — enable GPU runtime!")
# preflight: every runtime file the pipeline needs must exist in the clone —
# fail LOUDLY here rather than silently abandoning every unit in cell 4
REQUIRED = ["models/trained_head.joblib", "models/thresholds.json",
            "models/player_detector.pt", "models/court_grid_snapped.pt",
            "models/court_line_seg.pt", "data/harvest/games.json"]
missing = [f for f in REQUIRED if not os.path.exists(f)]
assert not missing, f"MISSING RUNTIME FILES (fix repo before running): {missing}"
print("preflight OK — all runtime files present")

In [ ]:
#@title 2 · Storage: Drive-live persistence (true mid-run resume)
import os
PERSIST = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST = "/content/drive/MyDrive/nba_harvest"
except Exception as e:
    print("Drive mount unavailable through this client:", e)
    PERSIST = "/content/nba_harvest"
    print("→ VM-local only: a disconnect LOSES all progress. Avoid for long runs.")
for d in ("video", "tracking", "results"):
    os.makedirs(f"{PERSIST}/{d}", exist_ok=True)
os.makedirs("data/harvest", exist_ok=True)
# LIVE persistence: videos, per-stage status, and section outputs all write
# straight to Drive — a disconnect mid-run costs at most one in-flight stage.
os.system(f"ln -sfn {PERSIST}/video data/harvest/video")
os.system(f"cp -rn data/tracking/. {PERSIST}/tracking/ 2>/dev/null")   # repo files (backups etc.)
os.system(f"rm -rf data/tracking && ln -sfn {PERSIST}/tracking data/tracking")
if not os.path.exists(f"{PERSIST}/status.json"):
    open(f"{PERSIST}/status.json", "w").write("{}")
os.system(f"ln -sf {PERSIST}/status.json data/harvest/status.json")
print("persist dir:", PERSIST, "(status + outputs live on it)")

In [ ]:
#@title 3 · Download games (skips ones already on Drive)
GAMES = ["gsw_cle_2017f_g5", "gsw_sac_klay37", "gsw_nyk_curry54",
         "gsw_mia_2017", "gsw_splash62", "gsw_hou_duel"]
import json, subprocess, os
reg = json.load(open("data/harvest/games.json"))
for t in GAMES:
    dst = f"data/harvest/video/{t}.mp4"
    if os.path.exists(dst) and os.path.getsize(dst) > 1e8:
        print(t, "already on Drive"); continue
    vid = reg[t]["video_id"]
    r = subprocess.run(["yt-dlp", "-f", "bv*[height<=720][height>=480]+ba",
                        "--merge-output-format", "mp4", "-o", dst, "-q", "--no-warnings",
                        f"https://www.youtube.com/watch?v={vid}"])
    print(t, "OK" if r.returncode == 0 else "FAILED — if YouTube blocks Colab IPs, "
          "upload the mp4s from your laptop to Drive:nba_harvest/video/ instead")

In [ ]:
#@title 4 · Run the harvest queue (videos LOCALIZED off Drive first)
# cv2 frame-seeking over the Drive FUSE mount fails silently (run-2 postmortem:
# every build read 0 frames and wrote {}). Videos are COPIED to VM-local disk
# for processing; Drive stays the durable download cache. Status + outputs stay
# on Drive (small JSON writes are FUSE-safe), so resume still works — a fresh
# VM just re-copies videos (~1-2 min/game).
import os, shutil, subprocess, glob
LOCAL_V = "/content/video_local"
os.makedirs(LOCAL_V, exist_ok=True)
for f in glob.glob(f"{PERSIST}/video/*.mp4"):
    dst = f"{LOCAL_V}/{os.path.basename(f)}"
    if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(f)):
        print("localizing", os.path.basename(f))
        shutil.copy(f, dst)
os.system(f"ln -sfn {LOCAL_V} data/harvest/video")
r = subprocess.run(["python", "harvest_driver.py", "--games"] + GAMES + ["--no-align"])
print("driver exit", r.returncode)
# show any failed units WITH their reasons — no more silent no-ops
import json as _j
st = _j.load(open("data/harvest/status.json"))
for c, u in sorted(st.items()):
    for stage, rec in u.items():
        if rec.get("state") != "ok":
            print("FAILED", c, stage, "→", (rec.get("tail") or "")[:150])

In [ ]:
#@title 5 · Align + join + credit (CPU-light, GPU OCR)
import json, subprocess
st = json.load(open("data/harvest/status.json"))
secs = sorted(c for c, u in st.items() if "_s" in c and u.get("segment", {}).get("state") == "ok"
              and any(c.startswith(t) for t in GAMES))
print(len(secs), "sections to align")
subprocess.run(["python", "align_outcomes.py", "--clips"] + secs)
for c in secs:
    subprocess.run(["python", "matchup_metrics.py", "--clip", c, "--no-video"],
                   capture_output=True)
subprocess.run(["python", "tier2_join.py"])
subprocess.run(["python", "tier2_credit.py"])

In [ ]:
#@title 6 · Package results
import os
os.system(f'cd {PERSIST} && zip -qr results/night3_tracking.zip tracking -i "tracking/gsw_*"')
os.system(f"zip -qr {PERSIST}/results/night3_pbp.zip data/pbp")
print("results in:", f"{PERSIST}/results")
if PERSIST.startswith("/content/nba_harvest"):
    print("NO DRIVE — download both zips via the file browser BEFORE the VM recycles.")
else:
    print("on Drive: nba_harvest/results/ — everything already persisted live anyway.")